In [1]:
import os
import re
import joblib
import numpy as np
import pandas as pd

from collections import Counter
from typing import List
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler

In [2]:
class TextPreprocessor(BaseEstimator, TransformerMixin):

    def clean_text(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.clean_text(text) for text in X]

In [3]:
class StatisticalFeatures(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for text in X:
            features.append([
                len(text),
                text.count("!"),
                text.count("?"),
                sum(word.isupper() for word in text.split())
            ])
        return np.array(features)

In [4]:
def build_dataset():
    texts = [
        "Invoice not generated",
        "Refund not received",
        "Incorrect billing amount",
        "Duplicate charge",
        "Subscription payment failed",
        "Billing issue urgent",
        "Server is down",
        "Application crashes on login",
        "Database timeout error",
        "System failure ASAP",
        "API not responding",
        "Bug in production environment",
        "Need contract review",
        "Legal compliance question",
        "NDA document request",
        "Data privacy concern",
        "Terms clarification",
        "Agreement amendment required"
    ]

    labels = [
        "Billing","Billing","Billing","Billing","Billing","Billing",
        "Technical","Technical","Technical","Technical","Technical","Technical",
        "Legal","Legal","Legal","Legal","Legal","Legal"
    ]

    return pd.DataFrame({"text": texts, "label": labels})


df = build_dataset()

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [6]:
class_counts = Counter(y_train)
min_class_size = min(class_counts.values())

n_splits = min(5, min_class_size)

print("Using n_splits =", n_splits)

Using n_splits = 4


In [7]:
word_tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=2000,
    stop_words="english"
)

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=1000
)

feature_union = FeatureUnion([
    ("word_features", word_tfidf),
    ("char_features", char_tfidf),
    ("stat_features", StatisticalFeatures())
])

In [8]:
base_svm = LinearSVC(class_weight="balanced")

In [9]:
advanced_pipeline = Pipeline([
    ("preprocess", TextPreprocessor()),
    ("features", feature_union),
    ("scaler", StandardScaler(with_mean=False)),
    ("classifier", base_svm)
])

In [10]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    advanced_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro"
)

print("CV F1:", cv_scores)
print("Mean:", np.mean(cv_scores))

CV F1: [0.61111111 0.33333333 0.38888889]
Mean: 0.4444444444444444


In [11]:
advanced_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess', TextPreprocessor()),
                ('features',
                 FeatureUnion(transformer_list=[('word_features',
                                                 TfidfVectorizer(max_features=2000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 stop_words='english')),
                                                ('char_features',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=1000,
                                                                 ngram_range=(3,
                                                                              5))),
                                                ('stat_features',
                                                 StatisticalFeatures())])),
                ('scaler', StandardScaler(with_mean=False)),
                ('classifier', LinearSVC(class_weight='balanced'))])

In [12]:
calibrated_model = CalibratedClassifierCV(
    advanced_pipeline,
    cv="prefit",
    method="sigmoid"
)

calibrated_model.fit(X_train, y_train)

c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


CalibratedClassifierCV(cv='prefit',
                       estimator=Pipeline(steps=[('preprocess',
                                                  TextPreprocessor()),
                                                 ('features',
                                                  FeatureUnion(transformer_list=[('word_features',
                                                                                  TfidfVectorizer(max_features=2000,
                                                                                                  ngram_range=(1,
                                                                                                               2),
                                                                                                  stop_words='english')),
                                                                                 ('char_features',
                                                                                  TfidfVectorizer(analyzer='char',
                                                                                                  max_features=1000,
                                                                                                  ngram_range=(3,
                                                                                                               5))),
                                                                                 ('stat_features',
                                                                                  StatisticalFeatures())])),
                                                 ('scaler',
                                                  StandardScaler(with_mean=False)),
                                                 ('classifier',
                                                  LinearSVC(class_weight='balanced'))]))

In [13]:
os.makedirs("model", exist_ok=True)

joblib.dump(
    advanced_pipeline,
    "model/mvr_classifier.joblib"
)

print("Baseline Classifier model saved.")

Baseline Classifier model saved.


In [14]:
class BaselineClassifier:
    
    def __init__(self, model_path: str):
        self.model = joblib.load(model_path)
    
    def predict(self, text: str):
        return self.model.predict([text])[0]
    
    def predict_proba(self, text: str):
        probs = self.model.predict_proba([text])[0]
        classes = self.model.classes_
        return dict(zip(classes, probs))

In [15]:
logistic_model = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=2000,
    class_weight="balanced"
)

In [16]:
advanced_pipeline = Pipeline([
    ("preprocess", TextPreprocessor()),
    ("features", feature_union),
    ("scaler", StandardScaler(with_mean=False)),
    ("classifier", logistic_model)
])

In [17]:
class_counts = Counter(y_train)
min_class_size = min(class_counts.values())

n_splits = min(3, min_class_size)  

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    advanced_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro"
)

print("CV F1 Scores:", cv_scores)
print("Mean CV F1:", np.mean(cv_scores))

CV F1 Scores: [0.38888889 0.33333333 0.38888889]
Mean CV F1: 0.3703703703703703


c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [18]:
advanced_pipeline.fit(X_train, y_train)

c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('preprocess', TextPreprocessor()),
                ('features',
                 FeatureUnion(transformer_list=[('word_features',
                                                 TfidfVectorizer(max_features=2000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 stop_words='english')),
                                                ('char_features',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=1000,
                                                                 ngram_range=(3,
                                                                              5))),
                                                ('stat_features',
                                                 StatisticalFeatures())])),
                ('scaler', StandardScaler(with_mean=False)),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    multi_class='multinomial'))])

In [19]:
y_pred = advanced_pipeline.predict(X_test)

print("Test F1 Score:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test F1 Score: 0.2222222222222222

Classification Report:

              precision    recall  f1-score   support

     Billing       0.00      0.00      0.00         1
       Legal       0.50      1.00      0.67         1
   Technical       0.00      0.00      0.00         2

    accuracy                           0.25         4
   macro avg       0.17      0.33      0.22         4
weighted avg       0.12      0.25      0.17         4



c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\kaviy\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [20]:
os.makedirs("model", exist_ok=True)

joblib.dump(
    advanced_pipeline,
    "model/mvr_logistic_classifier.joblib"
)

print("Logistic Classifier Model saved successfully.")

Logistic Classifier Model saved successfully.


In [21]:
class LogisticClassifier:

    def __init__(self, model_path):
        self.model = joblib.load(model_path)

    def predict(self, text):
        return self.model.predict([text])[0]

    def predict_proba(self, text):
        probs = self.model.predict_proba([text])[0]
        classes = self.model.classes_
        return dict(zip(classes, probs))